# Content Recommendation Engine
## Media & Entertainment ML Lab

### Business Context
For streaming platforms, personalized recommendations drive **engagement and retention**. Static editorial picks can't compete with ML-driven personalization that adapts to individual viewing patterns.

### What You'll Build
An end-to-end recommendation system using Databricks:

| Phase | What | Tools |
|-------|------|-------|
| **1. Feature Engineering** | Transform raw viewing history into ML-ready features | Delta Lake, Feature Engineering |
| **2. Model Training** | ALS collaborative filtering with hyperparameter tuning | PySpark ML, MLflow |
| **3. Real-Time Serving** | Deploy model for low-latency recommendation API calls | Model Serving, REST API |

### Architecture
```
Viewing History (Delta) → Feature Engineering → ALS Training (MLflow) → Model Registry (UC) → Serving Endpoint
```

### Prerequisites
- Run `00_Setup_Data_Generation` first to create the synthetic dataset

In [0]:
%pip install databricks-feature-engineering>=0.8.0 -q
dbutils.library.restartPython()

In [0]:
%run "./00_Setup_Data_Generation"

In [0]:
# --- Lab Configuration ---
SCHEMA = "content_reco_lab.content_reco_demo"
MODEL_NAME = f"{SCHEMA}.content_recommender_als"  # Unity Catalog model path
EXPERIMENT_NAME = "/Users/{}/content_reco_experiment".format(spark.sql("SELECT current_user()").first()[0])

# --- Imports ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import mlflow, warnings, logging
from mlflow.models import infer_signature
import mlflow.spark
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print(f"Schema: {SCHEMA}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Model: {MODEL_NAME}")

---
# Part 1: Feature Engineering with Delta Lake

Great recommendation systems depend on **high-quality features**. In this section, we:
1. Explore the raw viewing data to understand patterns
2. Engineer **user-level** features (viewing habits, genre preferences)
3. Engineer **content-level** features (popularity, avg ratings)
4. Create a **point-in-time correct** training dataset (preventing data leakage)

### Why Point-in-Time Correctness Matters
When building features for training, you must ensure features are computed using **only data available at prediction time** not future data. This prevents **data leakage** and ensures your model generalizes to production.

In [0]:
# Load the three source tables
df_users = spark.table(f"{SCHEMA}.users")
df_content = spark.table(f"{SCHEMA}.content_catalog")
df_views = spark.table(f"{SCHEMA}.viewing_history")

print("Dataset Overview")
print("=" * 50)
print(f"  Users:           {df_users.count():>8,}")
print(f"  Content Items:   {df_content.count():>8,}")
print(f"  Viewing Events:  {df_views.count():>8,}")

# Show date range of viewing history
date_range = df_views.agg(
    F.min("view_timestamp").alias("earliest"),
    F.max("view_timestamp").alias("latest"),
    F.avg("rating").alias("avg_rating"),
    F.avg("watch_pct").alias("avg_watch_pct")
).first()

print(f"\n  Date Range:      {date_range.earliest} → {date_range.latest}")
print(f"  Avg Rating:      {date_range.avg_rating:.2f}")
print(f"  Avg Watch %:     {date_range.avg_watch_pct:.1f}%")

print("Sample Users:")
display(df_users.limit(5))

In [0]:
# --- Exploratory Data Analysis ---
# Understand viewing patterns that will inform our feature engineering

# 1. Views by genre (which genres are most popular?)
df_genre_stats = (
    df_views
    .join(df_content, "content_id")
    .groupBy("genre")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("unique_viewers")
    )
    .orderBy(F.desc("total_views"))
)

print("Genre Popularity & Engagement:")
display(df_genre_stats)

# 2. Engagement by subscription tier
df_tier_stats = (
    df_views
    .join(df_users, "user_id")
    .groupBy("subscription_tier")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("active_users")
    )
    .withColumn("views_per_user", F.round(F.col("total_views") / F.col("active_users"), 1))
    .orderBy("subscription_tier")
)

print("Engagement by Subscription Tier:")
display(df_tier_stats)

In [0]:
# --- User-Level Feature Engineering ---
# These features capture each user's viewing behavior and preferences.
# In production, these would be maintained in the Databricks Feature Store.

# Define a "training cutoff" for point-in-time correctness
# We'll use data up to 2025-12-31 for features, hold out 2026+ for validation
TRAINING_CUTOFF = "2025-12-31"

df_views_train = df_views.filter(F.col("view_timestamp") <= TRAINING_CUTOFF)
df_views_test = df_views.filter(F.col("view_timestamp") > TRAINING_CUTOFF)

print(f"Training views (before {TRAINING_CUTOFF}): {df_views_train.count():,}")
print(f"Test views (after {TRAINING_CUTOFF}):      {df_views_test.count():,}")

# Enrich views with content metadata for feature engineering
df_views_enriched = df_views_train.join(df_content, "content_id")

# Step 1: Core user engagement features
df_user_features = (
    df_views_enriched
    .groupBy("user_id")
    .agg(
        # Engagement metrics
        F.count("*").alias("total_views"),
        F.countDistinct("content_id").alias("unique_content_viewed"),
        F.avg("watch_pct").alias("avg_watch_completion"),
        F.avg("rating").alias("avg_rating_given"),
        F.count(F.when(F.col("rating").isNotNull(), 1)).alias("num_ratings"),
        
        # Recency
        F.max("view_timestamp").alias("last_view_date"),
        F.datediff(F.lit(TRAINING_CUTOFF), F.max("view_timestamp")).alias("days_since_last_view"),
        
        # Genre diversity (how many distinct genres does the user watch?)
        F.countDistinct("genre").alias("genre_diversity")
    )
)

# Step 2: Binge indicator - max views per user per day
df_daily_views = (
    df_views_train
    .withColumn("view_date", F.to_date("view_timestamp"))
    .groupBy("user_id", "view_date")
    .agg(F.count("*").alias("daily_views"))
)

df_binge = (
    df_daily_views
    .groupBy("user_id")
    .agg(F.max("daily_views").alias("max_daily_views"))
)

# Step 3: Primary device (most frequently used device per user)
df_device_mode = (
    df_views_train
    .groupBy("user_id", "device_type")
    .agg(F.count("*").alias("device_count"))
)

w = Window.partitionBy("user_id").orderBy(F.desc("device_count"))
df_primary_device = (
    df_device_mode
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("user_id", F.col("device_type").alias("primary_device"))
)

# Combine all user features
df_user_features = (
    df_user_features
    .join(df_binge, "user_id", "left")
    .join(df_primary_device, "user_id", "left")
    .join(df_users.select("user_id", "subscription_tier", "region", "age"), "user_id")
)

print(f"\n\u2705 User features created for {df_user_features.count()} users")
display(df_user_features.limit(5))

In [0]:
# --- Content-Level Feature Engineering ---
# These features capture how popular and well-received each content item is.

df_content_features = (
    df_views_train
    .groupBy("content_id")
    .agg(
        # Popularity metrics
        F.count("*").alias("total_views"),
        F.countDistinct("user_id").alias("unique_viewers"),
        
        # Quality signals
        F.avg("rating").alias("avg_user_rating"),
        F.stddev("rating").alias("rating_stddev"),
        F.avg("watch_pct").alias("avg_completion_rate"),
        
        # Engagement depth
        F.count(F.when(F.col("watch_pct") > 80, 1)).alias("high_completion_count"),
        F.count(F.when(F.col("watch_pct") < 20, 1)).alias("early_dropout_count")
    )
    .withColumn("completion_ratio",
        F.round(F.col("high_completion_count") / F.col("total_views"), 3)
    )
)

# Join with content metadata
df_content_features = (
    df_content_features
    .join(df_content, "content_id")
)

print(f"Content features created for {df_content_features.count()} items")
print("Top 10 Most Viewed Content:")
display(
    df_content_features
    .select("title", "genre", "content_type", "total_views", "avg_user_rating", "avg_completion_rate")
    .orderBy(F.desc("total_views"))
    .limit(10)
)

In [0]:
# --- Point-in-Time Correct Training Dataset ---
# For collaborative filtering, our core input is the user-item interaction matrix.
# We use ONLY training-period data to prevent leakage.
#
# KEY CONCEPT: Point-in-time correctness means:
# - Features for user U are computed from views BEFORE the training cutoff
# - The target (rating/implicit feedback) is also from the training period
# - Test data (after cutoff) is reserved for evaluation only

# Create the interaction matrix for ALS
# ALS needs: user_id (int), content_id (int), rating (float)
df_interactions = (
    df_views_train
    .filter(F.col("rating").isNotNull())  # ALS explicit feedback needs ratings
    .groupBy("user_id", "content_id")
    .agg(
        F.avg("rating").alias("rating"),  # Average if multiple views
        F.count("*").alias("view_count"),
        F.avg("watch_pct").alias("avg_watch_pct")
    )
)

print(f"Training Interaction Matrix:")
print(f"Unique user-content pairs: {df_interactions.count():,}")
print(f"Unique users:             {df_interactions.select('user_id').distinct().count():,}")
print(f"Unique content items:     {df_interactions.select('content_id').distinct().count():,}")
print(f"Sparsity:                 {1 - df_interactions.count() / (500 * 200):.2%}")

# Save feature tables as Delta for reproducibility
df_user_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_features")
df_content_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.content_features")
df_interactions.write.mode("overwrite").saveAsTable(f"{SCHEMA}.training_interactions")

print(f"Feature tables saved to {SCHEMA}")
print("  → user_features")
print("  → content_features")
print("  → training_interactions")

display(df_interactions.limit(10))

   
---
# Part 2: Model Training with ALS & MLflow

### What is ALS? (The Simple Version)

Imagine a giant spreadsheet where every **row is a user** and every **column is a show or movie**. Each cell holds that user's rating but almost every cell is **blank**, because no one has time to watch everything!

Our data doesn't actually look like that giant grid, it's stored as a simple list of entries like *(User 42 rated Movie 7 → 4.5 stars)*. But ALS **thinks** of it as that big grid and tries to **fill in all the blanks**, predicting what rating each user *would* give to shows they haven't seen yet.

How does it do that? It figures out hidden patterns (called **factors**), things like *"this user loves sci-fi"* and *"this show is very sci-fi"*. If a user's taste lines up with a show's qualities, ALS predicts a high rating. It works by going back and forth:
1. First it locks the user patterns and figures out the show patterns
2. Then it locks the show patterns and figures out the user patterns
3. It repeats this over and over until the predictions stop improving

That back-and-forth is the **"Alternating"** part of Alternating Least Squares!

### Key Knobs We Can Turn (Hyperparameters)
| Knob | What It Controls | Range We'll Try |
|-----------|-------------|---------------|
| `rank` | How many hidden taste/quality patterns to look for (more = more nuanced, but slower) | 10 – 100 |
| `maxIter` | How many back-and-forth rounds to run | 10 – 15 |
| `regParam` | How much we prevent the model from overthinking (overfitting) the training data | 0.01 – 0.5 |

### MLflow Integration
We use MLflow to:
- **Track** each combination of knobs and how well it performed (RMSE)
- **Compare** runs side-by-side in the MLflow UI
- **Register** the winning model to Unity Catalog so it can be deployed to production

In [0]:
# --- Prepare ALS Input ---
# Load the interaction matrix we saved earlier
df_als_input = spark.table(f"{SCHEMA}.training_interactions")

# Split into train/validation (80/20) for hyperparameter tuning
# Using random split since our point-in-time split is already handled
(als_train, als_val) = df_als_input.randomSplit([0.8, 0.2], seed=42)

print(f"ALS Training set:   {als_train.count():,} interactions")
print(f"ALS Validation set: {als_val.count():,} interactions")

# Preview the interaction data
print("Rating Distribution:")
display(
    df_als_input
    .withColumn("rating_bucket", F.round("rating", 0))
    .groupBy("rating_bucket")
    .count()
    .orderBy("rating_bucket")
)

In [0]:
# --- Hyperparameter Tuning with MLflow ---
# We'll run a grid search, logging each combination to MLflow

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Set up MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Define hyperparameter grid
param_grid = {
    "rank": [10, 50, 100],
    "maxIter": [10, 15],
    "regParam": [0.01, 0.1, 0.5]
}

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

best_rmse = float("inf")
best_model = None
best_params = {}
run_count = 0
total_runs = len(param_grid["rank"]) * len(param_grid["maxIter"]) * len(param_grid["regParam"])

print(f"Starting hyperparameter search: {total_runs} combinations")
print("=" * 60)

for rank in param_grid["rank"]:
    for maxIter in param_grid["maxIter"]:
        for regParam in param_grid["regParam"]:
            run_count += 1
            run_name = f"ALS_r{rank}_i{maxIter}_reg{regParam}"
            
            with mlflow.start_run(run_name=run_name) as run:
                # Log parameters
                mlflow.log_param("rank", rank)
                mlflow.log_param("maxIter", maxIter)
                mlflow.log_param("regParam", regParam)
                mlflow.log_param("algorithm", "ALS")
                mlflow.log_param("feedback_type", "explicit")
                
                # Train ALS model
                als = ALS(
                    rank=rank,
                    maxIter=maxIter,
                    regParam=regParam,
                    userCol="user_id",
                    itemCol="content_id",
                    ratingCol="rating",
                    coldStartStrategy="drop",  # Drop NaN predictions for new users/items
                    nonnegative=True  # Ratings are positive
                )
                
                model = als.fit(als_train)
                
                # Evaluate on validation set
                predictions = model.transform(als_val)
                rmse = evaluator.evaluate(predictions)
                
                # Log metrics
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("num_user_factors", model.userFactors.count())
                mlflow.log_metric("num_item_factors", model.itemFactors.count())
                
                # Track best model
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model = model
                    best_params = {"rank": rank, "maxIter": maxIter, "regParam": regParam}
                    best_run_id = run.info.run_id
                
                status = " ⭐ BEST" if rmse == best_rmse else ""
                print(f"  [{run_count}/{total_runs}] {run_name}: RMSE={rmse:.4f}{status}")

print("\n" + "=" * 60)
print(f"Best Model: RMSE={best_rmse:.4f}")
print(f" Parameters: {best_params}")
print(f"Run ID: {best_run_id}")

In [0]:
# --- Evaluate the Best Model ---
print(f"Best ALS Model Performance")
print(f"RMSE: {best_rmse:.4f}")
print(f"Params: rank={best_params['rank']}, maxIter={best_params['maxIter']}, regParam={best_params['regParam']}")

# Generate top-10 recommendations for all users
# Using transform() + window ranking instead of recommendForAllUsers()
# to avoid higher-order functions unsupported on UC serverless
df_all_users = df_als_input.select("user_id").distinct()
df_all_content = df_als_input.select("content_id").distinct()
df_all_pairs = df_all_users.crossJoin(df_all_content)

predictions = best_model.transform(df_all_pairs).filter(F.col("prediction").isNotNull())

w = Window.partitionBy("user_id").orderBy(F.desc("prediction"))
df_recs_exploded = (
    predictions
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 10)
    .select(
        "user_id",
        "rank",
        "content_id",
        F.round("prediction", 2).alias("predicted_rating")
    )
    .join(df_content.select("content_id", "title", "genre"), "content_id")
)

print("Sample Recommendations (User 1):")
display(
    df_recs_exploded
    .filter(F.col("user_id") == 1)
    .orderBy("rank")
)

# Save recommendations for later use
df_recs_exploded.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_recommendations")
print(f"\n\u2705 Recommendations saved to {SCHEMA}.user_recommendations")

In [0]:
# --- Recommendation Quality Check ---
# Verify our recommendations make sense by comparing them
# to users' known genre preferences

# For each user, check what % of recommendations match their preferred genres
df_quality = (
    df_recs_exploded
    .join(df_users.select("user_id", "preferred_genres"), "user_id")
    .withColumn("genre_match", 
        F.when(
            F.col("preferred_genres").contains(F.col("genre")), 1
        ).otherwise(0)
    )
    .groupBy("user_id")
    .agg(
        F.avg("genre_match").alias("preference_match_rate"),
        F.avg("predicted_rating").alias("avg_predicted_score"),
        F.count("*").alias("num_recs")
    )
)

overall_match = df_quality.agg(F.avg("preference_match_rate")).first()[0]

print(f"🎯 Recommendation Quality Analysis")
print("=" * 60)
print(f"  Genre Preference Match Rate: {overall_match:.1%}")
print(f"  (% of recommended content matching user's preferred genres)")
print(f"  Random baseline would be ~30% (3 of 10 genres)")
print(f"  Our model achieves {overall_match:.1%} → {'✅ Above random!' if overall_match > 0.30 else '⚠️ Needs improvement'}")

print("\n📊 Match Rate Distribution:")
display(
    df_quality
    .withColumn("match_bucket", 
        F.when(F.col("preference_match_rate") >= 0.8, "80-100%")
        .when(F.col("preference_match_rate") >= 0.6, "60-80%")
        .when(F.col("preference_match_rate") >= 0.4, "40-60%")
        .when(F.col("preference_match_rate") >= 0.2, "20-40%")
        .otherwise("0-20%")
    )
    .groupBy("match_bucket")
    .agg(F.count("*").alias("num_users"))
    .orderBy("match_bucket")
)

In [0]:
# The ALS model is logged to UC for versioning, lineage, and reproducibility.
# Serving is handled by Feature Serving (Part 3), not by a model serving endpoint.
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# UC Volume for temporary Spark model storage (required on serverless)
VOLUME_PATH = f"/Volumes/content_reco_lab/content_reco_demo/ml_artifacts"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.ml_artifacts")

sample_input = als_train.select("user_id", "content_id").limit(5).toPandas()

with mlflow.start_run(run_name="best_als_model_registration") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("rmse", best_rmse)

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="als_model",
        registered_model_name=MODEL_NAME,
        input_example=sample_input,
        dfs_tmpdir=f"{VOLUME_PATH}/tmp",
    )

    print(f"\n\u2705 Model registered to Unity Catalog: {MODEL_NAME}")
    print(f"   Run ID: {run.info.run_id}")

from mlflow import MlflowClient
client = MlflowClient()
latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if latest_versions:
    latest_version = max(int(v.version) for v in latest_versions)
    client.set_registered_model_alias(MODEL_NAME, "Champion", str(latest_version))
    print(f"   Alias 'Champion' set to version {latest_version}")

   
   
---
# Part 3: Real-Time Serving with Feature Serving

### Why Not a Model Serving Endpoint?

Our ALS model is built with PySpark, which needs a full **Java Virtual Machine (JVM)** to load. Model Serving containers are lightweight, CPU-constrained environments (4 GB per concurrency unit) — not designed to spin up Spark. We *could* wrap the model in a custom pyfunc class that extracts the factor matrices and does dot-product math in pure NumPy, but there's a simpler option that's actually **faster and cheaper**.

### The Key Insight: Recommendations Are Lookups, Not Computations

Think about how a streaming app actually uses recommendations:

| Widget | What the app needs | How often it changes |
|--------|-------------------|---------------------|
| **"Top 10 For You"** | Given a `user_id`, return their best content | Once per retrain (nightly) |
| **"Because You Watched X"** | Given a `content_id`, return similar items | Once per retrain (nightly) |

Both are **static lookups** — the answers only change when the model is retrained. So instead of running a model on every request, we:
1. **Precompute** all the answers in batch (cells 12–13)
2. **Store** them in wide-format Delta tables (1 row per key)
3. **Sync** them to a low-latency **Online Store** (Lakebase)
4. **Serve** them via a **Feature Serving endpoint** as fast key-value lookups

### Architecture
```
  ┌──────────────┐     ┌───────────────────┐     ┌───────────────┐     ┌──────────────┐
  │  ALS Model    │────▶│  Delta Tables      │────▶│  Online Store  │────▶│  REST API     │
  │  (Spark, on   │     │  (wide format,     │     │  (Lakebase,    │     │  (Feature     │
  │   cluster)    │     │   1 row per key)   │     │   auto-sync)   │     │   Serving)    │
  └──────────────┘     └───────────────────┘     └───────────────┘     └──────────────┘
       Batch                 Cell 13                  Cell 16               Cell 17
     (nightly)           PK + CDF enabled           publish_table        <10ms lookups
```

### Two Precomputed Tables

**`serving_user_recs`** — "Top 10 For You"  
One row per `user_id` with 10 recommendation slots (`rec_1_content_id`, `rec_1_title`, `rec_1_score`, ... `rec_10_*`).  
Built from the ALS model's predicted ratings across all user–item pairs (cell 12), then pivoted to wide format (cell 13).

**`serving_item_similarities`** — "Because You Watched X"  
One row per `content_id` with 10 similar-item slots (`sim_1_content_id`, `sim_1_title`, `sim_1_score`, ... `sim_10_*`).  
Built by computing **cosine similarity** between ALS item factor vectors (cell 13). Items with similar latent patterns get high scores.

### The "Just Watched" Scenario
When a user finishes a show, the app sends **one request** with both `user_id` and `content_id`. The endpoint returns **both** sets of recommendations in a single response — personalized picks *and* similar items — in under 10 ms.

### Benefits Over Model Serving
| | Model Serving (pyfunc) | Feature Serving (this approach) |
|---|---|---|
| **Custom code needed** | Yes — pyfunc class, factor extraction, pickle artifacts | No — just Delta tables |
| **Latency** | \~50–100 ms (dot-product compute) | <10 ms (key-value lookup) |
| **Cold start** | Slow (model loading) | None (data is pre-synced) |
| **Refresh cycle** | Redeploy model version | Just refresh Delta tables |
| **Cost** | Pay for model compute per request | Pay for online store sync |

In [0]:
# --- Prepare Feature Serving Tables ---
# Instead of deploying a model serving endpoint, we precompute recommendations
# and serve them as low-latency key-value lookups via Feature Serving.
#
# Two tables (wide format: 1 row per key):
#   1. serving_user_recs:          user_id    → "Top 10 For You"
#   2. serving_item_similarities:  content_id → "Because You Watched X"

import numpy as np
import pandas as pd

# --- Step 1: Compute Item-to-Item Cosine Similarity ---
print("Step 1: Computing item-to-item similarities from ALS factors...")
item_factors_pdf = best_model.itemFactors.select("id", "features").toPandas()
item_ids = item_factors_pdf["id"].values.astype(int)
item_matrix = np.stack(item_factors_pdf["features"].values).astype(np.float32)

# Cosine similarity
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
item_normed = item_matrix / np.maximum(norms, 1e-10)
sim_matrix = item_normed @ item_normed.T

# Content metadata lookup
content_pdf = df_content.select("content_id", "title", "genre").toPandas()
id_to_meta = {
    int(r["content_id"]): (r["title"], r["genre"])
    for _, r in content_pdf.iterrows()
}

# Build wide-format table (1 row per content_id, top-10 similar items)
TOP_N = 10
sim_rows = []
for i, cid in enumerate(item_ids):
    sims = sim_matrix[i].copy()
    sims[i] = -1  # exclude self
    top_idx = np.argsort(sims)[::-1][:TOP_N]

    row = {"content_id": int(cid)}
    for rank, idx in enumerate(top_idx, 1):
        sid = int(item_ids[idx])
        title, genre = id_to_meta.get(sid, ("Unknown", "Unknown"))
        row[f"sim_{rank}_content_id"] = sid
        row[f"sim_{rank}_title"] = title
        row[f"sim_{rank}_genre"] = genre
        row[f"sim_{rank}_score"] = round(float(sims[idx]), 4)
    sim_rows.append(row)

df_item_sim = spark.createDataFrame(pd.DataFrame(sim_rows))
df_item_sim.write.mode("overwrite").saveAsTable(f"{SCHEMA}.serving_item_similarities")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities ALTER COLUMN content_id SET NOT NULL")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities DROP CONSTRAINT IF EXISTS item_sim_pk")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities ADD CONSTRAINT item_sim_pk PRIMARY KEY (content_id)")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')")
print(f"  \u2705 {SCHEMA}.serving_item_similarities ({len(sim_rows)} items \u00d7 top-{TOP_N})")

display(df_item_sim)

In [0]:
# --- Step 2: Pivot User Recommendations to Wide Format ---
print("\nStep 2: Pivoting user recommendations to wide format...")
recs_pdf = spark.table(f"{SCHEMA}.user_recommendations").toPandas()
rec_rows = []
for uid, group in recs_pdf.groupby("user_id"):
    group = group.sort_values("rank")
    row = {"user_id": int(uid)}
    for _, r in group.head(TOP_N).iterrows():
        rank = int(r["rank"])
        row[f"rec_{rank}_content_id"] = int(r["content_id"])
        row[f"rec_{rank}_title"] = r["title"]
        row[f"rec_{rank}_genre"] = r["genre"]
        row[f"rec_{rank}_score"] = round(float(r["predicted_rating"]), 4)
    rec_rows.append(row)

df_user_recs = spark.createDataFrame(pd.DataFrame(rec_rows))
df_user_recs.write.mode("overwrite").saveAsTable(f"{SCHEMA}.serving_user_recs")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs ALTER COLUMN user_id SET NOT NULL")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs DROP CONSTRAINT IF EXISTS user_recs_pk")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs ADD CONSTRAINT user_recs_pk PRIMARY KEY (user_id)")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')")
print(f"  \u2705 {SCHEMA}.serving_user_recs ({len(rec_rows)} users \u00d7 top-{TOP_N})")

# --- Preview ---
print("\n\U0001f4cb Sample: Top-3 similar items for content_id=1:")
display(df_user_recs)

### Databricks Services Used in This Cell

The cell below orchestrates several Databricks platform services to stand up a real-time recommendation API without writing any custom model-inference code:

| Service | What It Does Here | Why We Need It |
|---|---|---|
| **Feature Engineering Client** (`FeatureEngineeringClient`) | Creates feature specs and publishes Delta tables to the online store | Provides the high-level API that ties offline tables to online serving — no manual plumbing required |
| **Online Store (Lakebase)** | Hosts a low-latency, key-value copy of our two Delta tables | Delta tables are optimized for analytics, not point lookups. Lakebase gives us <10 ms reads by syncing data into a format built for single-row fetches |
| **Feature Serving Endpoint** | Exposes a managed REST API backed by the online store | Lets any application (mobile app, web backend, etc.) retrieve recommendations over HTTPS with auto-scaling and scale-to-zero |
| **Unity Catalog** | Stores the Feature Spec (`content_recommender_spec`) and governs access to the underlying tables | Centralizes metadata, lineage, and permissions so every asset — tables, specs, and endpoints — is discoverable and auditable in one place |
| **Databricks SDK** (`WorkspaceClient`) | Creates and updates the serving endpoint programmatically | Allows us to manage infrastructure-as-code: check if an endpoint exists, update its config, or create a new one — all from within the notebook |
| **Delta Change Data Feed (CDF)** | Enabled on the source tables in the previous cell | Lets the online store incrementally sync only the rows that changed, making nightly refreshes fast and cost-efficient |

### 🗤️ What Is an Online Store? What Is Lakebase?

**Delta tables are built for analytics** — scanning millions of rows, running aggregations, training models. They're *not* designed for single-row, sub-10ms lookups that a production app needs when a user opens their homepage.

That's where the **Online Feature Store** comes in. It's a separate, high-performance serving layer purpose-built for **point lookups**: given a key (like `user_id`), return that one row instantly.

**Lakebase** is the engine that powers it. Under the hood, it's a fully managed **Postgres database** integrated into the Databricks platform. When you "publish" a Delta table to an online store, Databricks syncs the data into Lakebase so it can be read at operational speed. Key characteristics:

| | Delta Table (Offline) | Lakebase (Online) |
|---|---|---|
| **Optimized for** | Analytics, batch, ML training | Single-row lookups, OLTP |
| **Latency** | Seconds to minutes | Sub-10 ms |
| **Access pattern** | Scan millions of rows | Fetch one row by primary key |
| **Kept in sync via** | — | Change Data Feed (CDF) from the source Delta table |

In our case, the two Delta tables (`serving_user_recs` and `serving_item_similarities`) are synced into Lakebase so that when the Feature Serving endpoint receives a request, it reads from Lakebase — not from Delta — to get that <10 ms response time.

> **Think of it this way:** Delta is your warehouse. Lakebase is the vending machine stocked from that warehouse — fast to access, limited to what's been loaded, and automatically restocked when the source changes.

In [0]:
# --- Feature Serving Endpoint ---
# Serve precomputed recommendations via low-latency key-value lookup.
# No custom model, no pyfunc class — just table lookups!
#
# Architecture:
#   Delta Table → Online Store (Lakebase) → Feature Spec → REST Endpoint
#
# One endpoint, two lookups:
#   • user_id    → "Top 10 For You"        (personalized recs)
#   • content_id → "Because You Watched X"  (similar items)

# Re-define constants (Python was restarted for package install)
SCHEMA = "content_reco_lab.content_reco_demo"
ENDPOINT_NAME = "content-recommender-endpoint"

from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

fe = FeatureEngineeringClient()
w = WorkspaceClient()

ONLINE_STORE_NAME = "content-reco-online-store"
FEATURE_SPEC_NAME = f"{SCHEMA}.content_recommender_spec"

# --- Step 1: Create Online Store ---
print("Step 1: Creating online store...")
try:
    fe.create_online_store(name=ONLINE_STORE_NAME, capacity="CU_1")
    print(f"  \u2705 Created: {ONLINE_STORE_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"  \u2139\ufe0f  Already exists: {ONLINE_STORE_NAME}")
    else:
        raise

### 📤 What Does Publishing a Table Do?

`publish_table` takes a Delta table from the lakehouse and **syncs its data into Lakebase** (the online store) so it can be read at sub-10ms latency. Behind the scenes, Databricks creates a **DLT pipeline** that handles the data movement.

### Do I need to republish when the source table changes?

It depends on the **sync mode** you choose via the `publish_mode` parameter:

| Mode | Behavior | When source data changes… |
|---|---|---|
| **TRIGGERED** (default) | One-time incremental sync using CDF | You must **re-trigger** the sync to pick up changes |
| **CONTINUOUS** | Streaming pipeline runs continuously | Changes propagate to Lakebase **automatically** in near real-time |
| **SNAPSHOT** | Full copy of the entire table each time | You must **re-trigger** — the whole table is replaced |

Our notebook doesn’t specify `publish_mode`, so it defaults to **TRIGGERED**. That means after a nightly retrain updates the Delta tables, the online store will **not** update itself — you need to re-trigger the sync.

### How do I re-trigger?

You have three options:

1. **Rerun this cell** — Yes, simply rerunning `publish_table` works. It detects the existing synced table and performs an incremental sync (only changed rows, thanks to CDF).
2. **Trigger the underlying pipeline** — `publish_table` returns an object with a `pipeline_id`. You can trigger that pipeline directly via the API or UI without rerunning this notebook.
3. **Schedule it in a Databricks Job** — Wrap the `publish_table` call in a job that runs after the nightly retrain step. This is the recommended production pattern.

> **Tip:** If you want zero-touch updates, pass `publish_mode="CONTINUOUS"` to `publish_table`. The online store will then stream changes from the Delta table automatically — but at a higher cost.

In [0]:
# --- Step 2: Publish Feature Tables to Online Store ---
# NOTE: online_table_name must differ from source_table_name (use ot_ prefix)
print("\nStep 2: Publishing tables to online store...")
online_store = fe.get_online_store(name=ONLINE_STORE_NAME)

publish_configs = {
    "serving_user_recs": "ot_serving_user_recs",
    "serving_item_similarities": "ot_serving_item_similarities",
}

for source_table, online_table in publish_configs.items():
    print(f"  Publishing {source_table}...")
    try:
        fe.publish_table(
            online_store=online_store,
            source_table_name=f"{SCHEMA}.{source_table}",
            online_table_name=f"{SCHEMA}.{online_table}",
        )
        print(f"  \u2705 Published: {source_table} \u2192 {online_table}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  \u2139\ufe0f  Already published: {online_table}")
        else:
            raise

### 📋 What Is a Feature Spec?

A **Feature Spec** is a Unity Catalog object that defines *which tables to look up and which keys to use* when serving data. Think of it as a **wiring diagram** for the Feature Serving endpoint.

In our case, the spec says:

| Lookup Key | Source Table | What It Returns |
|---|---|---|
| `user_id` | `serving_user_recs` | That user's top-10 personalized recommendations |
| `content_id` | `serving_item_similarities` | The top-10 items similar to that content |

When a request hits the endpoint with `{user_id: 42, content_id: 147}`, the Feature Spec tells the serving layer: *"Look up user 42 in the first table and content 147 in the second table, then combine and return the results."*

The spec itself holds no data — it's purely metadata that maps keys → tables → columns.

In [0]:
# --- Step 3: Create Feature Spec ---
print("\nStep 3: Creating feature spec...")
features = [
    FeatureLookup(
        table_name=f"{SCHEMA}.serving_user_recs",
        lookup_key="user_id",
    ),
    FeatureLookup(
        table_name=f"{SCHEMA}.serving_item_similarities",
        lookup_key="content_id",
    ),
]

try:
    fe.create_feature_spec(name=FEATURE_SPEC_NAME, features=features)
    print(f"  \u2705 Created: {FEATURE_SPEC_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"  \u2139\ufe0f  Already exists: {FEATURE_SPEC_NAME}")
    else:
        raise

### 💡 Why Feature Serving Instead of Querying Lakebase Directly?

You might be wondering: *"We already synced our Delta tables to Lakebase (the online store). Why can't we just query Lakebase directly instead of standing up a Feature Serving endpoint?"*

Great question — here's the key point:

**Lakebase doesn't expose its own REST API.** It's a storage layer optimized for low-latency key-value reads, but it has no standalone HTTP interface that an external application can call. Think of it like a fast internal database with no front door.

The **Feature Serving endpoint is that front door.** It provides:

| What | Why it matters |
|---|---|
| **Managed REST API** | Any app (mobile, web, backend service) can call it over HTTPS |
| **Authentication & governance** | Integrated with Unity Catalog — access control and audit out of the box |
| **Auto-scaling & scale-to-zero** | Handles traffic spikes without manual infra management, costs nothing at idle |
| **Single request, multiple lookups** | One call with `{user_id, content_id}` returns *both* personalized recs and similar items |

### Isn't "Feature Serving" misleading? These aren't really features…

You're right — in the traditional ML sense, "features" are inputs to a model for real-time prediction. Here, our recommendations are **precomputed outputs**, not model inputs. No inference happens at request time.

We're essentially borrowing the Feature Serving infrastructure as a **general-purpose, low-latency lookup API**. Databricks designed this path specifically for cases like ours — the [docs](https://docs.databricks.com/machine-learning/feature-store/feature-serving.html) call them "precomputed features," but the pattern works equally well for any precomputed result you want to serve at low latency.

**TL;DR:** Lakebase = fast storage engine (no API). Feature Serving = managed REST gateway on top of that storage. You need both.

In [0]:
# --- Step 4: Create/Update Serving Endpoint ---
print(f"\nStep 4: Setting up endpoint '{ENDPOINT_NAME}'...")
served_entity = ServedEntityInput(
    entity_name=FEATURE_SPEC_NAME,
    scale_to_zero_enabled=True,
    workload_size="Small",
)

try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"  Endpoint exists (state: {existing.state.ready}). Updating...")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[served_entity]
    )
    print(f"  \u2705 Endpoint updated: {ENDPOINT_NAME}")
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        print(f"  Creating endpoint (5-10 minutes)...")
        w.serving_endpoints.create_and_wait(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                name=ENDPOINT_NAME,
                served_entities=[served_entity]
            )
        )
        print(f"  \u2705 Endpoint READY: {ENDPOINT_NAME}")
    else:
        raise

print(f"\n\U0001f389 Feature Serving is live!")
print(f"   Endpoint: {ENDPOINT_NAME}")
print(f"   Send {{user_id, content_id}} \u2192 get personalized recs + similar items")

In [0]:
# --- Test Feature Serving Endpoint ---
# Scenario: User 42 just finished watching content 147 (a Drama).
# The app calls the endpoint with {user_id: 42, content_id: 147} and gets:
#   • "Top 10 For You" — personalized recommendations for user 42
#   • "Because You Watched" — items similar to content 147

import mlflow.deployments

ENDPOINT_NAME = "content-recommender-endpoint"
client = mlflow.deployments.get_deploy_client("databricks")

# Simulate: "User 42 just watched content 147"
response = client.predict(
    endpoint=ENDPOINT_NAME,
    inputs={
        "dataframe_records": [
            {"user_id": 42, "content_id": 147}
        ]
    },
)

result = response["outputs"][0]

print("\U0001f3ac User 42 just watched content 147")
print("=" * 60)

# Display personalized recommendations
print("\n\U0001f4cb Top 10 For You (personalized):")
for i in range(1, 11):
    title = result.get(f"rec_{i}_title", "\u2014")
    genre = result.get(f"rec_{i}_genre", "\u2014")
    score = result.get(f"rec_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (score: {score})")

# Display similar items
print(f"\n\U0001f517 Because You Watched Content 147:")
for i in range(1, 11):
    title = result.get(f"sim_{i}_title", "\u2014")
    genre = result.get(f"sim_{i}_genre", "\u2014")
    score = result.get(f"sim_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (similarity: {score})")

print(f"\n\u2705 Feature Serving: <10ms latency, no model inference \u2014 just lookups!")

### In production, a nightly Databricks Job refreshes recommendations:
  1. Ingest new viewing data from streaming pipeline
  2. Retrain ALS model (or incrementally update factors)
  3. Regenerate user_recommendations (cell 12)
  4. Regenerate serving tables (cell 13)
  5. Re-publish to Online Store → Feature Serving auto-updates

The Feature Serving endpoint always serves the latest published data.
No model redeployment needed — just refresh the underlying tables.

In [0]:
# --- OPTIONAL: Full Cleanup ---
# Removes ALL resources created by this notebook AND 00_Setup_Data_Generation.
# WARNING: This is irreversible. Uncomment the line at the bottom to execute.

def cleanup_lab():
    from databricks.sdk import WorkspaceClient
    from databricks.feature_engineering import FeatureEngineeringClient
    from mlflow import MlflowClient
    import mlflow

    SCHEMA = "content_reco_lab.content_reco_demo"
    CATALOG = "content_reco_lab"
    MODEL_NAME = f"{SCHEMA}.content_recommender_als"
    ENDPOINT_NAME = "content-recommender-endpoint"
    ONLINE_STORE_NAME = "content-reco-online-store"
    FEATURE_SPEC_NAME = f"{SCHEMA}.content_recommender_spec"
    EXPERIMENT_NAME = "/Users/{}/content_reco_experiment".format(
        spark.sql("SELECT current_user()").first()[0]
    )

    w = WorkspaceClient()
    fe = FeatureEngineeringClient()
    mlflow.set_registry_uri("databricks-uc")
    ml_client = MlflowClient()

    errors = []

    def safe(label, fn):
        try:
            fn()
            print(f"  \u2705 {label}")
        except Exception as e:
            if "does not exist" in str(e).lower() or "not found" in str(e).lower() or "resource_does_not_exist" in str(e).upper():
                print(f"  \u2139\ufe0f  {label} (already gone)")
            else:
                errors.append((label, e))
                print(f"  \u26a0\ufe0f  {label}: {e}")

    print("\U0001f9f9 Starting full lab cleanup...")
    print("=" * 60)

    # 1. Delete serving endpoint (depends on feature spec)
    print("\n[1/7] Serving Endpoint")
    safe(f"Delete endpoint '{ENDPOINT_NAME}'",
         lambda: w.serving_endpoints.delete(ENDPOINT_NAME))

    # 2. Delete feature spec from Unity Catalog
    print("\n[2/7] Feature Spec")
    safe(f"Delete feature spec '{FEATURE_SPEC_NAME}'",
         lambda: fe.delete_feature_spec(name=FEATURE_SPEC_NAME))

    # 3. Drop published online tables, then delete the online store
    print("\n[3/7] Online Store & Published Tables")
    online_tables = [
        f"{SCHEMA}.ot_serving_user_recs",
        f"{SCHEMA}.ot_serving_item_similarities",
    ]
    online_store = None
    try:
        online_store = fe.get_online_store(name=ONLINE_STORE_NAME)
    except Exception:
        pass

    if online_store:
        for ot in online_tables:
            safe(f"Drop online table '{ot}'",
                 lambda ot=ot: fe.drop_online_table(name=ot, online_store=online_store))
        safe(f"Delete online store '{ONLINE_STORE_NAME}'",
             lambda: fe.delete_online_store(name=ONLINE_STORE_NAME))
    else:
        print(f"  \u2139\ufe0f  Online store '{ONLINE_STORE_NAME}' (already gone)")

    # 4. Delete registered model (all versions)
    print("\n[4/7] MLflow Model")
    def delete_model():
        versions = ml_client.search_model_versions(f"name='{MODEL_NAME}'")
        for v in versions:
            ml_client.delete_model_version(MODEL_NAME, v.version)
        ml_client.delete_registered_model(MODEL_NAME)
    safe(f"Delete model '{MODEL_NAME}'", delete_model)

    # 5. Delete MLflow experiment and all runs
    print("\n[5/7] MLflow Experiment")
    def delete_experiment():
        exp = ml_client.get_experiment_by_name(EXPERIMENT_NAME)
        if exp:
            ml_client.delete_experiment(exp.experiment_id)
        else:
            raise Exception("not found")
    safe(f"Delete experiment '{EXPERIMENT_NAME}'", delete_experiment)

    # 6. Drop schema CASCADE (removes ALL tables + volume)
    #    Tables: users, content_catalog, viewing_history (setup notebook)
    #            user_features, content_features, training_interactions,
    #            user_recommendations, serving_user_recs,
    #            serving_item_similarities (main notebook)
    #    Volume: ml_artifacts
    print("\n[6/7] Schema (all tables + volumes)")
    safe(f"Drop schema '{SCHEMA}' CASCADE",
         lambda: spark.sql(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE"))

    # 7. Drop catalog
    print("\n[7/7] Catalog")
    safe(f"Drop catalog '{CATALOG}'",
         lambda: spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE"))

    # Summary
    print("\n" + "=" * 60)
    if errors:
        print(f"\u26a0\ufe0f  Cleanup finished with {len(errors)} issue(s):")
        for label, e in errors:
            print(f"   - {label}: {e}")
    else:
        print("\u2705 All lab resources cleaned up successfully!")

# \u26a0\ufe0f Uncomment the line below to run the full cleanup:
# cleanup_lab()